[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barmag/fanous-llm-lens/blob/main/notebooks/education/stage1_c_subword_experiment.ipynb)

# Stage 1c: Zero-Layer Transformer (Subword/BPE-Level) 🧩
## target: Is "dialect" a direction you can find in learned embeddings?

We counted characters (Stage 1a) and word transitions (Stage 1b). Now we take the
last step of the embeddings trilogy: instead of *counting*, we **learn a vector for
every subword** by training a tiny model on our own MSA + Masri text — then we ask one
sharp question about those vectors.

### 💡 What we are showing
1. Train a small **BPE tokenizer** on a mixed MSA+Masri corpus (no giant download — the
   subwords come from *our* data).
2. Train a **zero-layer transformer** — literally `embedding → unembedding → softmax over
   the next token`. This is a subword *bigram* predictor, and its embedding matrix `W_E`
   is the thing we inspect.
3. Ask: **is dialect linearly encoded in `W_E`?** We compare two views of the *same*
   vectors —
   - **Unsupervised PCA**, which surfaces the *loudest* variance (frequency/length) and
     leaves the dialect colours mixed, versus
   - **A supervised probe**, which finds the *one axis* that separates MSA from Masri and
     reports an accuracy number.

> ### ⚠️ Up front — what this is *not*
> These are **toy embeddings** from a one-matrix model trained on a small corpus. They
> capture co-occurrence, not deep meaning. Rich contextual semantics only emerge once a
> model has real layers — that is a *later* stage. The lesson here is the method:
> **PCA shows what is biggest; a probe shows what you asked for.**

In [ ]:
# 🛠️ Setup: Install dependencies if running on Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q tokenizers datasets scikit-learn plotly torch

## 📚 Step 1 — Build subwords from our own dialect data
We reuse the Stage 1b corpus: **Modern Standard Arabic** from Wikipedia and **Egyptian
(Masri)** from real street tweets. We train a small **BPE** tokenizer on the two combined,
then encode each stream into subword ids. Keeping the two streams separate matters — the
per-stream counts are exactly how we will label each subword's dialect later.

In [ ]:
# 📦 Fetch MSA + Masri text, then learn a BPE vocabulary from it
# TODO:
#   1. Use the HuggingFace datasets library to stream wikimedia/wikipedia
#      ("20231101.ar") for MSA and fetch amgadhasan/arabic_tweets_dialects
#      filtered to dialect == "EG" for Masri.
#   2. Clean each: collapse whitespace, strip non-Arabic with the regex range
#      [^\s\u0621-\u064A]; build msa_text and masri_text (~500k chars each).
#   3. Train a BPE tokenizer (tokenizers.Tokenizer + BpeTrainer, vocab_size=3000,
#      Whitespace pre-tokenizer) on [msa_text, masri_text] combined.
#   4. Produce msa_ids, masri_ids (tokenizer.encode(...).ids) and
#      id_to_token = {i: t for t, i in tokenizer.get_vocab().items()}.
#   5. Print the vocab size and a few example tokenizations (الذي, اللي, دلوقتي).

## 🧠 Step 2 — Train the zero-layer transformer
A **zero-layer transformer** has no attention and no MLP: a token's embedding goes
straight to the unembedding to predict the next token. That makes it a **subword bigram
model**. We train it with cross-entropy on `(current → next)` pairs pooled from both
streams; the learned embedding matrix `W_E` (shape `[vocab, 64]`) is our artefact.

In [ ]:
# 🏋️ Train embedding (W_E) + unembedding (W_U) to predict the next subword
# TODO:
#   1. Seed torch / numpy / random for reproducibility.
#   2. Build (current, next) subword pairs *within* each stream (don't bridge streams).
#   3. Define a zero-layer model: torch.nn.Embedding(vocab, 64) -> torch.nn.Linear(64,
#      vocab, bias=False). No attention, no MLP — that's the whole model.
#   4. Train until the loss roughly plateaus (~15 epochs; 3 is far too few)
#      with Adam + CrossEntropyLoss; print the falling loss.
#   5. Expose W_E = embed.weight.detach().numpy()  # shape [vocab, 64]

## 🔍 Step 3 — PCA vs. a dialect probe
**Free labels.** Every subword's dialect comes from the data itself: count how often it
appears in the MSA vs Masri stream. Mostly-MSA → `MSA`, mostly-Masri → `Masri`, roughly
even → `Shared` (rare tokens are dropped).

**Two views of the same `W_E`:**
- **PCA** (unsupervised) plots the two directions of largest variance. It does not know
  about dialect, so the colours tend to stay mixed — it shows what is *biggest*.
- **A linear probe** (logistic regression) is *told* the labels and finds the single axis
  that best separates MSA from Masri. We project every subword onto that axis and read off
  a held-out **accuracy**. `Shared` tokens are excluded from the probe entirely — never
  trained or tested on, only projected here, where they should land in the middle.

In [ ]:
# 🎨 Dialect labels, PCA, and a supervised probe — side by side
# TODO:
#   1. dialect_labels(msa_ids, masri_ids): count each id per stream; label MSA / Masri /
#      Shared by the MSA share (e.g. >=0.7 -> MSA, <=0.3 -> Masri), drop rare ids
#      and the [UNK] catch-all token.
#   2. PCA: sklearn PCA(n_components=2) on the labelled rows of W_E -> scatter, coloured
#      by dialect. Notice the colours stay MIXED (unsupervised shows the biggest variance).
#   3. Probe: train sklearn LogisticRegression on W_E rows (MSA vs Masri; hold out Shared),
#      report held-out accuracy, take clf.coef_[0] as the dialect axis, and project every
#      token onto it -> an outline (step) curve per dialect (no fills, so they
#      never blend) with the decision boundary marked.
#   4. Render with make_subplots(rows=1, cols=3): the PCA scatter, the outline
#      curves, and a strip plot (one lane per dialect) — then fig.show().
#   The point: PCA can't see dialect, but the probe can.

## ✅ Recap & hand-off
- We **learned** a vector per subword by training a one-matrix (zero-layer) model on our
  own MSA + Masri text — completing the trilogy: *count chars → count words → learn vectors*.
- **PCA** of those vectors leaves the dialects mixed: the loudest variance is frequency and
  length, not dialect.
- A **linear probe**, told the labels, recovers a dialect axis with well-above-chance held-out
  accuracy — so **dialect is linearly encoded even in a zero-layer model trained on mixed
  data**. The takeaway is the method: *PCA shows what is biggest; a probe shows what you
  asked for.*

**Next:** with real attention + MLP layers, embeddings start carrying *meaning*, not just
co-occurrence. That is where contextual semantics — and the MSA↔Masri story inside the
residual stream — pick up in the next stage.